In [1]:
!pip install pytorch-tabnet autogluon lightgbm -q
print("✅ 설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 12.0 MB/s eta 0:00

In [3]:
import torch
print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from google.colab import drive
drive.mount('/content/drive')

GPU: False
Device: CPU
Mounted at /content/drive


In [17]:
import os, random, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pytorch_tabnet.tab_model import TabNetClassifier
import autogluon.tabular
print("✅ autogluon OK:", autogluon.tabular.__version__)

# ===============================================================
# 0️⃣ Config / Reproducibility
# ===============================================================
SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 21
print(f"✅ Using device: {DEVICE}")

# 데이터 경로 설정
DATA_DIR = "/content/drive/MyDrive/머신러닝/data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# ===============================================================
# 1️⃣ Dataset Wrapper
# ===============================================================
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X.astype(np.float32)
        self.y = None if y is None else y.astype(np.int64)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

✅ autogluon OK: 1.5.0
✅ Using device: cpu


In [6]:
# ===============================================================
# 🆕 Feature Engineering (자동 핵심 센서 5개 선정 + 캐시)
# - 데이터 특성 반영: 52센서, 중복 쌍 많음, 스케일 차이 큼 (X_19=50 vs X_01=0.018)
# - RF importance ∩ MI 교집합 우선 + 상관 0.75 이상 제외
# - 통계 계산 전 z-score 정규화 필수
# ===============================================================
import os, json
from itertools import combinations

# CACHE_DIR이 아직 정의 안 됐을 때 대비 (Cell 15보다 먼저 실행될 수 있음)
if 'CACHE_DIR' not in globals():
    CACHE_DIR = os.path.join(DATA_DIR, "cache")
    os.makedirs(CACHE_DIR, exist_ok=True)
CORE_CACHE_PATH = os.path.join(CACHE_DIR, "core_sensors.json")


def _select_core_sensors(df, y, sensor_cols, k=5, n_top=10, corr_thr=0.75):
    """RF + MI 교집합 우선, 상관 0.75 이상 제외 (중복 센서 방지)"""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.feature_selection import mutual_info_classif

    X = df[sensor_cols].values

    print("  🌲 RF importance 계산 중...")
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=15, n_jobs=-1, random_state=SEED
    )
    rf.fit(X, y)
    rf_imp = rf.feature_importances_

    print("  📊 Mutual information 계산 중...")
    mi = mutual_info_classif(X, y, random_state=SEED, n_jobs=-1)

    sensor_arr = np.array(sensor_cols)
    top_rf = set(sensor_arr[np.argsort(-rf_imp)[:n_top]])
    top_mi = set(sensor_arr[np.argsort(-mi)[:n_top]])

    # 교집합 우선 (RF importance 순)
    intersect = [s for s in sensor_arr[np.argsort(-rf_imp)]
                 if s in (top_rf & top_mi)]
    union_extra = [s for s in sensor_arr[np.argsort(-rf_imp)]
                   if s in (top_rf | top_mi) and s not in intersect]
    candidates = intersect + union_extra

    # 상관 그리디 필터 (0.75 이상이면 제외)
    corr_mat = df[candidates].corr().abs()
    selected = [candidates[0]]
    for c in candidates[1:]:
        if len(selected) >= k:
            break
        if corr_mat.loc[c, selected].max() < corr_thr:
            selected.append(c)

    # 부족하면 corr 무시하고 채움
    if len(selected) < k:
        for c in candidates:
            if c not in selected and len(selected) < k:
                selected.append(c)

    print(f"🎯 Auto-selected {k} core sensors: {selected}")
    print(f"   RF∩MI top-{n_top}: {sorted(top_rf & top_mi)}")
    print(f"   상관 매트릭스:\n{df[selected].corr().round(2)}")
    return selected


def get_or_select_core_sensors(df, y=None, k=5, force=False):
    """캐시 있으면 로드, 없으면 선정 후 캐시 저장"""
    if os.path.exists(CORE_CACHE_PATH) and not force:
        with open(CORE_CACHE_PATH) as f:
            cached = json.load(f)
        if cached.get('k') == k:
            print(f"📦 Load cached core sensors (k={k}): {cached['sensors']}")
            return cached['sensors']
        else:
            print(f"⚠️ Cached k={cached.get('k')} != requested k={k}, 재선정 중...")

    if y is None:
        raise ValueError(
            "❌ Core sensors 캐시 없음. train df + y로 먼저 호출하세요."
        )

    sensor_cols = [c for c in df.columns if c.startswith('X_')]
    selected = _select_core_sensors(df, y, sensor_cols, k=k)

    with open(CORE_CACHE_PATH, 'w') as f:
        json.dump({'sensors': selected, 'k': k}, f)
    print(f"💾 Cached to {CORE_CACHE_PATH}")
    return selected


def feature_engineering(df, y=None, k=5):
    """
    Args:
        df: input dataframe
        y: train일 때만 (test는 None → 캐시 사용)
        k: 핵심 센서 개수 (이 데이터엔 5 권장)
    """
    df = df.copy()
    sensors = [c for c in df.columns if c.startswith('X_')]

    # 기존: row-wise skew (LDA가 못 잡는 비선형 통계)
    df['row_skew'] = df[sensors].skew(axis=1)

    # 자동 선정된 core sensors
    core = get_or_select_core_sensors(df, y=y, k=k)

    # ⭐⭐ 스케일 차이가 3000배 (X_19 vs X_01) → z-score 정규화 후 통계 ⭐⭐
    core_z = df[core].apply(lambda c: (c - c.mean()) / (c.std() + 1e-8))

    df['c_mean']  = core_z.mean(axis=1)
    df['c_std']   = core_z.std(axis=1)
    df['c_max']   = core_z.max(axis=1)
    df['c_min']   = core_z.min(axis=1)
    df['c_range'] = df['c_max'] - df['c_min']

    return df

In [7]:
# ===============================================================
# 2️⃣ MLP Model (ReLU / GELU)
# ===============================================================
class ResidualBlock(nn.Module):
    """Residual block with normalization and activation."""
    def __init__(self, dim, dropout=0.2, act="relu"):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
    def forward(self, x):
        residual = x
        out = self.act_fn(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        out = self.act_fn(out + residual)
        return out

class MLPEncoderWithHead(nn.Module):
    """Encoder + projection + classification head"""
    def __init__(self, input_dim, act="relu", num_classes=NUM_CLASSES):
        super().__init__()
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act),
            nn.Linear(256, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act)
        )
        self.embedding = nn.Linear(256, 128)
        self.projection = nn.Sequential(nn.Linear(128, 128), self.act_fn, nn.Linear(128, 128))
        self.classifier = nn.Sequential(nn.LayerNorm(128), self.act_fn, nn.Linear(128, num_classes))
    def forward(self, x):
        h = self.backbone(x)
        emb = self.embedding(h)
        z = self.projection(F.normalize(emb, dim=1))
        logits = self.classifier(emb)
        return emb, z, logits

In [8]:

# ===============================================================
# 3️⃣ Loss Functions
# ===============================================================
class SupConLoss(nn.Module):
    """Supervised Contrastive Loss"""
    def __init__(self, temperature=0.1):
        super().__init__(); self.temperature = temperature
    def forward(self, features, labels):
        device = features.device; batch_size = features.shape[0]
        labels = labels.contiguous().view(-1,1)
        mask = torch.eq(labels, labels.T).float().to(device)
        anchor_dot = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot, dim=1, keepdim=True)
        logits = anchor_dot - logits_max.detach()
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=device)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True)+1e-12)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1)+1e-12)
        loss = -mean_log_prob_pos
        return loss.mean()

class CenterLoss(nn.Module):
    """Center Loss: improves intra-class compactness"""
    def __init__(self, num_classes, feat_dim, lambda_c=1.0):
        super().__init__()
        self.register_buffer('centers', torch.randn(num_classes, feat_dim))
        self.lambda_c = lambda_c
    def forward(self, emb, labels):
        centers_batch = self.centers[labels]
        return self.lambda_c * ((emb - centers_batch)**2).sum() / (2.0 * emb.size(0))
    @torch.no_grad()
    def update_centers(self, emb, labels, lr=0.5):
        for c in torch.unique(labels):
            mask = labels==c
            if mask.sum()==0:
                continue
            mean = emb[mask].mean(dim=0)
            self.centers[c] = (1-lr)*self.centers[c] + lr*mean

In [9]:
# ===============================================================
# 🆕 LDA Feature Augmentation Helper (fold-safe)
# ===============================================================
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

class LDAFeatureAugmenter:
    """
    Fold-safe LDA projection augmenter.
    - fit() : train fold에만 LDA + post-scaler fit
    - transform() : val/test에 transform 후 concat

    Returns concatenated [raw_scaled | lda_projection_scaled]
    """
    def __init__(self, n_components=None, shrinkage='auto'):
        self.n_components = n_components
        self.shrinkage = shrinkage
        self.lda = None
        self.post_scaler = None  # LDA output 표준화용

    def fit(self, X_scaled, y, num_classes):
        # n_components는 min(n_features, n_classes-1) 이하
        max_comp = min(X_scaled.shape[1], num_classes - 1)
        n_comp = max_comp if self.n_components is None else min(self.n_components, max_comp)

        self.lda = LinearDiscriminantAnalysis(
            n_components=n_comp,
            solver='eigen',          # ⭐ shrinkage 가능한 solver
            shrinkage=self.shrinkage # ⭐ Ledoit-Wolf 자동 shrinkage
        )
        self.lda.fit(X_scaled, y)

        # ⭐ LDA 출력은 스케일이 들쭉날쭉 → 표준화
        lda_proj = self.lda.transform(X_scaled)
        self.post_scaler = StandardScaler().fit(lda_proj)
        return self

    def transform(self, X_scaled):
        lda_proj = self.lda.transform(X_scaled)
        lda_proj = self.post_scaler.transform(lda_proj)
        return np.hstack([X_scaled, lda_proj])

    @property
    def out_dim_info(self):
        return f"raw={self.lda.n_features_in_}, lda={self.lda.n_components}"


def diagnose_lda(lda_aug, X_val_scaled, y_val):
    """LDA가 실제로 분리력 있는지 빠르게 확인 (logistic regression sanity check)"""
    from sklearn.linear_model import LogisticRegression
    proj = lda_aug.lda.transform(X_val_scaled)
    # validation 한정 빠른 score (참고용)
    lr = LogisticRegression(max_iter=200, n_jobs=-1).fit(proj, y_val)
    return lr.score(proj, y_val)

In [10]:
# ===============================================================
# 4️⃣ MLP Training
# ===============================================================
def evaluate_model(model, loader):
    """Evaluate model accuracy & F1"""
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _, _, logits = model(xb)
            preds.append(torch.softmax(logits, dim=1).cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds, trues = np.vstack(preds), np.concatenate(trues)
    acc = accuracy_score(trues, preds.argmax(1))
    f1 = f1_score(trues, preds.argmax(1), average="macro")
    return acc, f1, preds

def train_mlp_model(X, y, X_test, act="relu", epochs=100, use_lda=True):  # ⭐ flag 추가
    from sklearn.utils.class_weight import compute_class_weight
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    # ⭐ input_dim을 LDA 적용 후 차원으로 계산하기 위해 미리 한 fold 시뮬
    # (또는 fold 안에서 동적으로 model 생성하면 됨 — 이미 그렇게 되어있음)

    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds  = np.zeros((len(X), NUM_CLASSES))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔥 Fold {fold} | act={act} | use_lda={use_lda}")

        # === 기존 scaling ===
        fold_scaler = StandardScaler()
        X_tr_s  = fold_scaler.fit_transform(X[tr_idx])
        X_val_s = fold_scaler.transform(X[val_idx])
        X_te_s  = fold_scaler.transform(X_test)

        # ⭐⭐⭐ LDA augmentation ⭐⭐⭐
        if use_lda:
            lda_aug = LDAFeatureAugmenter(n_components=20, shrinkage='auto')
            lda_aug.fit(X_tr_s, y[tr_idx], num_classes=NUM_CLASSES)
            X_tr_s  = lda_aug.transform(X_tr_s)
            X_val_s = lda_aug.transform(X_val_s)
            X_te_s  = lda_aug.transform(X_te_s)

            # 진단 출력 (선택, 첫 fold만)
            if fold == 0:
            # 원본(LDA 적용 전) val 데이터로 진단해야 함
                X_val_raw_only = X_val_s[:, :X[tr_idx].shape[1]]
                sanity = diagnose_lda(lda_aug, X_val_raw_only, y[val_idx])
                print(f"  📐 LDA dims: {lda_aug.out_dim_info} | sanity LR acc: {sanity:.4f}")


        # === 나머지 기존 코드 그대로 ===
        train_loader = DataLoader(TabularDataset(X_tr_s,  y[tr_idx]), batch_size=256, shuffle=True, drop_last=True)
        val_loader   = DataLoader(TabularDataset(X_val_s, y[val_idx]), batch_size=512, shuffle=False)
        test_loader  = DataLoader(TabularDataset(X_te_s), batch_size=512, shuffle=False)

        # ⭐ input_dim이 raw+20 = X.shape[1]+20으로 자동 반영됨
        model = MLPEncoderWithHead(input_dim=X_tr_s.shape[1], act=act).to(DEVICE)

        # ... 이하 기존 코드 (optimizer, scheduler, loss, training loop) 동일 ...

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
        # ⭐ [B] 클래스 불균형 보정 (train fold만 사용)
        cw = compute_class_weight('balanced',
                                  classes=np.arange(NUM_CLASSES),
                                  y=y[tr_idx])
        cw_t = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
        ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05, weight=cw_t)

        supcon = SupConLoss(0.1)
        center = CenterLoss(NUM_CLASSES, 128, 1.0).to(DEVICE)

        #시간 단축, 다른 곳에 시간 투자 -early stop
        best_f1, best_state = -1, None
        patience = 20          # 20 epoch 동안 개선 없으면 중단
        no_improve = 0
        for ep in range(epochs):
            model.train(); losses=[]
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                emb, z, logits = model(xb)
                loss = 0.5*supcon(z,yb) + 1.0*ce_loss(logits,yb) + 0.1*center(emb,yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                center.update_centers(emb.detach(), yb.detach())
                losses.append(loss.item())
            acc, f1, _ = evaluate_model(model, val_loader)
            scheduler.step(1-f1)
            print(f"Epoch {ep}: loss={np.mean(losses):.4f} val_f1={f1:.4f}")
            if f1 > best_f1:
                best_f1 = f1
                # ⭐ clone()으로 깊은 복사 - 이게 핵심 버그 픽스
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"⏹️ Early stop at epoch {ep}, best_f1={best_f1:.4f}")
                    break

        model.load_state_dict(best_state)
        torch.save(model.state_dict(), f"{MODEL_DIR}/mlp_{act}_fold{fold}.pth")

        # ⭐ [A] validation 예측 → OOF 저장
        _, _, val_proba = evaluate_model(model, val_loader)
        oof_preds[val_idx] = val_proba

        model.eval()

        all_test=[]
        with torch.no_grad():
            for xb in test_loader:
                xb = xb.to(DEVICE)
                _,_,logits=model(xb)
                all_test.append(torch.softmax(logits,dim=1).cpu().numpy())
        test_preds += np.vstack(all_test)/skf.n_splits
    return test_preds, oof_preds

In [11]:
# ===============================================================
# 5️⃣ TabNet
# ===============================================================
from pytorch_tabnet.pretraining import TabNetPretrainer

def train_tabnet_kfold(X, y, X_test, n_splits=5, use_pretrain=True, use_lda=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds  = np.zeros((len(X), NUM_CLASSES))
    fold_scores = []

    # ⭐ Pretraining은 LDA 없이 raw로 (test 데이터 활용이 목적)
    pretrainer = None
    if use_pretrain:
        print("🧪 TabNet self-supervised pretraining (raw features)...")
        # StandardScaler 한 번 적용 (전체 데이터 — pretraining은 unsupervised라 OK)
        # 하지만 정합성 위해 fold 외부 scaler를 별도로 운영
        pre_scaler = StandardScaler().fit(X)
        X_unlab = np.vstack([pre_scaler.transform(X), pre_scaler.transform(X_test)])
        pretrainer = TabNetPretrainer(
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=2e-2),
            mask_type='entmax',
            n_d=32, n_a=32, n_steps=4, gamma=1.5,
            seed=SEED, verbose=0,
        )
        pretrainer.fit(
            X_train=X_unlab,
            max_epochs=100, patience=20,
            batch_size=2048, virtual_batch_size=256,
            pretraining_ratio=0.8,
        )

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🌲 TabNet Fold {fold+1}/{n_splits} | use_lda={use_lda}")

        # ⭐ fold-safe scaling (기존 TabNet 코드엔 없었지만 LDA 위해 필요)
        fold_scaler = StandardScaler()
        X_tr  = fold_scaler.fit_transform(X[tr_idx])
        X_val = fold_scaler.transform(X[val_idx])
        X_te  = fold_scaler.transform(X_test)

        if use_lda:
            lda_aug = LDAFeatureAugmenter(n_components=20, shrinkage='auto')
            lda_aug.fit(X_tr, y[tr_idx], num_classes=NUM_CLASSES)
            X_tr  = lda_aug.transform(X_tr)
            X_val = lda_aug.transform(X_val)
            X_te  = lda_aug.transform(X_te)

        y_tr, y_val = y[tr_idx], y[val_idx]

        # ⭐⭐ pretrainer dimension mismatch 주의 ⭐⭐
        # pretrainer는 raw dim으로 학습됨. LDA augmented input은 dim이 다름!
        # 해결책: LDA 쓸 때는 from_unsupervised=None으로 (pretrain 포기)
        # 또는: pretrain도 LDA augmented input으로 다시 학습 (더 좋지만 비쌈)
        use_pretrainer_this_fold = pretrainer if not use_lda else None

        model = TabNetClassifier(
            n_d=32, n_a=32, n_steps=4, gamma=1.5,
            lambda_sparse=1e-4,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=2e-2),
            scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts,
            scheduler_params=dict(T_0=10, T_mult=1, eta_min=1e-5),
            mask_type='entmax',
            seed=SEED + fold,
            verbose=10,
        )
        model.fit(
            X_train=X_tr, y_train=y_tr,
            eval_set=[(X_val, y_val)],
            eval_name=['val'],
            eval_metric=['accuracy', 'logloss'],
            max_epochs=300, patience=40,
            batch_size=1024, virtual_batch_size=128,
            num_workers=2, drop_last=False,
            from_unsupervised=use_pretrainer_this_fold,  # ⭐ LDA면 None
            weights=1,
        )
        # ... 이하 동일 ...

        # 저장
        model.save_model(os.path.join(MODEL_DIR, f"tabnet_fold{fold}"))

        # 예측
        val_proba = model.predict_proba(X_val)
        oof_preds[val_idx] = val_proba
        test_preds += model.predict_proba(X_te) / n_splits

        f1 = f1_score(y_val, val_proba.argmax(1), average='macro')
        fold_scores.append(f1)
        print(f"  Fold {fold+1} macro-F1: {f1:.4f}")

    print(f"\n📊 TabNet CV mean F1: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
    return test_preds, oof_preds

In [12]:
# ===============================================================
# 6️⃣ AutoGluon
# ===============================================================
def train_autogluon(train_df, test_df, use_lda=True):
    from autogluon.tabular import TabularPredictor
    label = 'target'

    if use_lda:
        # ⭐ AutoGluon은 내부 CV를 하므로, 여기선 전체 train에 LDA fit
        # (AutoGluon stacking이 자체적으로 holdout 관리)
        feature_cols = [c for c in train_df.columns if c != label]
        X_tr = train_df[feature_cols].values
        y_tr = train_df[label].values
        X_te = test_df[feature_cols].values

        scaler = StandardScaler().fit(X_tr)
        X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

        lda_aug = LDAFeatureAugmenter(n_components=20, shrinkage='auto')
        lda_aug.fit(X_tr_s, y_tr, num_classes=NUM_CLASSES)
        lda_tr = lda_aug.lda.transform(X_tr_s)
        lda_tr = lda_aug.post_scaler.transform(lda_tr)
        lda_te = lda_aug.lda.transform(X_te_s)
        lda_te = lda_aug.post_scaler.transform(lda_te)

        # 컬럼명 부여 → AutoGluon이 feature importance 추적 가능
        lda_cols = [f"lda_{i}" for i in range(lda_tr.shape[1])]
        train_df = pd.concat([
            train_df.reset_index(drop=True),
            pd.DataFrame(lda_tr, columns=lda_cols)
        ], axis=1)
        test_df = pd.concat([
            test_df.reset_index(drop=True),
            pd.DataFrame(lda_te, columns=lda_cols)
        ], axis=1)

    predictor = TabularPredictor(
        label=label,
        eval_metric='f1_macro',
        path=os.path.join(MODEL_DIR, "autogluon")
    ).fit(
        train_data=train_df,
        time_limit=3600,
        presets='best_quality'
    )
    return predictor.predict_proba(test_df).values

In [15]:
# ===============================================================
# 🌳 LightGBM 5-Fold (5번째 친구)
# ===============================================================
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler

def train_lightgbm_kfold(X, y, X_test, n_splits=5):
    """LightGBM 5-fold OOF + Test 예측"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_preds = np.zeros((len(X), NUM_CLASSES))
    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    fold_scores = []

    # 통합 정규화 (Covariate Shift 대응)
    scaler = StandardScaler()
    combined = np.vstack([X, X_test])
    scaler.fit(combined)
    X_scaled = scaler.transform(X)
    X_test_scaled = scaler.transform(X_test)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            objective='multiclass',
            num_class=NUM_CLASSES,
            random_state=SEED,
            verbose=-1,
            n_jobs=-1
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )

        oof_preds[val_idx] = model.predict_proba(X_val)
        test_preds += model.predict_proba(X_test_scaled) / n_splits

        f1 = f1_score(y_val, oof_preds[val_idx].argmax(1), average='macro')
        fold_scores.append(f1)
        print(f"  Fold {fold}: F1 = {f1:.4f}")

    print(f"📊 LightGBM 평균 OOF F1: {np.mean(fold_scores):.4f}")
    return test_preds, oof_preds

print("✅ LightGBM 함수 정의 완료")

✅ LightGBM 함수 정의 완료


In [13]:
# ===============================================================
# 7️⃣ Hungarian / Greedy Assignment
# ===============================================================
def compute_quota(probs):
    N,C=probs.shape; base=N//C; rem=N%C
    quota=np.full(C,base,int)
    strength=probs.mean(0); order=np.argsort(-strength)
    quota[order[:rem]]+=1
    return quota

def balanced_assign_hungarian(probs, quota):
    eps=1e-15; N,C=probs.shape
    cols=[]; col_to_class=[]
    for c in range(C):
        q=int(quota[c])
        cost_c=-np.log(probs[:,c]+eps)
        cols.append(np.repeat(cost_c[:,None],q,axis=1))
        col_to_class.extend([c]*q)
    cost=np.concatenate(cols,axis=1)
    row_ind,col_ind=linear_sum_assignment(cost)
    pred=np.zeros(N,int); pred[row_ind]=np.array(col_to_class)[col_ind]
    return pred

def balanced_assign_greedy(probs, quota):
    N,C=probs.shape; remaining=quota.copy(); pred=-np.ones(N,int)
    order=np.argsort(-probs.max(1)); ranked=np.argsort(-probs,1)
    for idx in order:
        for c in ranked[idx]:
            if remaining[c]>0:
                pred[idx]=c; remaining[c]-=1; break
    if (pred<0).any():
        for idx in np.where(pred<0)[0]:
            c=int(np.argmax(remaining)); pred[idx]=c; remaining[c]-=1
    return pred


from scipy.optimize import minimize

def find_best_weights(oof_list, y_true):
    """OOF로 최적 ensemble weight 탐색 (macro F1 최대화)"""
    def neg_f1(w):
        w = np.abs(w); w = w / w.sum()
        blend = sum(wi * oof for wi, oof in zip(w, oof_list))
        return -f1_score(y_true, blend.argmax(1), average='macro')
    init = np.ones(len(oof_list)) / len(oof_list)
    res = minimize(neg_f1, init, method='Nelder-Mead',
                   options={'xatol':1e-4, 'maxiter':500})
    w = np.abs(res.x); w = w / w.sum()
    return w, -res.fun

In [14]:
##Run all 들어가기 전 캐시 저장
# ===============================================================
# 🔖 Cache helper (Run all 들어가기 전 캐시 저장/로드)
# ===============================================================
CACHE_DIR = os.path.join(DATA_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_or_run(name, fn, *args, force=False, **kwargs):
    """
    name 기반으로 결과(.npz)를 저장/로드.
    force=True면 캐시 무시하고 재학습.
    fn은 tuple(np.ndarray, ...)을 리턴해야 함.
    """
    path = os.path.join(CACHE_DIR, f"{name}.npz")
    if os.path.exists(path) and not force:
        print(f"📦 Load cache: {path}")
        data = np.load(path, allow_pickle=True)
        return tuple(data[k] for k in data.files)
    print(f"🛠️  Run & cache: {name}")
    out = fn(*args, **kwargs)
    if not isinstance(out, tuple):
        out = (out,)
    np.savez(path, **{f"arr_{i}": a for i, a in enumerate(out)})
    return out

In [18]:
# ===============================================================
# 8️⃣ Run All (LightGBM 5번째 친구 추가)
# ===============================================================
def run_all():
    assert os.path.exists(TRAIN_PATH), f"❌ train.csv not found: {TRAIN_PATH}"
    assert os.path.exists(TEST_PATH),  f"❌ test.csv not found: {TEST_PATH}"
    print(f"✅ Data files OK")

    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)

    # Feature Engineering (Core 센서 자동 선정 포함)
    train = feature_engineering(train, y=train['target'].values, k=5)
    test = feature_engineering(test, k=5)
    print(f"✅ FE 적용 완료 | train: {train.shape}, test: {test.shape}")

    X, y = train.drop(columns=["target","ID"]).values, train["target"].values
    X_test = test.drop(columns=["ID"]).values

    # 1. TabNet
    print("\n🚀 [1/5] Training TabNet (5-Fold)...")
    tab_test, tab_oof = cache_or_run(
        "tabnet", train_tabnet_kfold, X, y, X_test, n_splits=5, use_pretrain=True
    )

    # 2. MLP-ReLU
    print("\n🚀 [2/5] Training MLP (ReLU)...")
    relu_test, relu_oof = cache_or_run(
        "mlp_relu", train_mlp_model, X, y, X_test, act="relu"
    )

    # 3. MLP-GELU
    print("\n🚀 [3/5] Training MLP (GELU)...")
    gelu_test, gelu_oof = cache_or_run(
        "mlp_gelu", train_mlp_model, X, y, X_test, act="gelu"
    )

    # 4. ⭐ LightGBM (새 친구)
    print("\n🚀 [4/5] Training LightGBM (5-Fold)...")
    lgb_test, lgb_oof = cache_or_run(
        "lightgbm", train_lightgbm_kfold, X, y, X_test, n_splits=5
    )

    # 5. AutoGluon
    print("\n🚀 [5/5] Training AutoGluon...")
    (autogluon_test,) = cache_or_run(
        "autogluon", train_autogluon,
        train.drop(columns=["ID"]), test.drop(columns=["ID"])
    )

    # ============================================
    # ⭐ 4명 자동 가중치 (LightGBM 포함) + AG 0.2
    # ============================================
    print("\n⚖️ Auto-weight ensemble (4 models + AG)")
    w, best = find_best_weights(
        [tab_oof, relu_oof, gelu_oof, lgb_oof], y
    )
    print(f"🎯 Weights: TabNet={w[0]:.3f} ReLU={w[1]:.3f} GELU={w[2]:.3f} LGB={w[3]:.3f} | OOF F1={best:.4f}")

    blend_4 = w[0]*tab_test + w[1]*relu_test + w[2]*gelu_test + w[3]*lgb_test
    ensemble = 0.8*blend_4 + 0.2*autogluon_test

    # Hungarian 후처리
    quota = compute_quota(ensemble)
    try:
        preds = balanced_assign_hungarian(ensemble, quota)
        method = "Hungarian"
    except Exception:
        preds = balanced_assign_greedy(ensemble, quota)
        method = "Greedy"

    submission = pd.DataFrame({"ID": test["ID"], "target": preds})
    output_path = os.path.join(DATA_DIR, "submission_5models.csv")
    submission.to_csv(output_path, index=False)
    print(f"\n✅ Final saved: {output_path} ({method} used)")
    print(f"\n📊 클래스별 예측 개수:")
    print(pd.Series(preds).value_counts().sort_index())


# Entry Point
if __name__ == "__main__":
    run_all()

✅ Data files OK
  🌲 RF importance 계산 중...
  📊 Mutual information 계산 중...
🎯 Auto-selected 5 core sensors: [np.str_('X_40'), np.str_('X_46'), np.str_('X_36'), np.str_('X_42'), np.str_('X_28')]
   RF∩MI top-10: [np.str_('X_28'), np.str_('X_33'), np.str_('X_36'), np.str_('X_40'), np.str_('X_41'), np.str_('X_42'), np.str_('X_46')]
   상관 매트릭스:
      X_40  X_46  X_36  X_42  X_28
X_40  1.00 -0.01  0.24 -0.14  0.19
X_46 -0.01  1.00 -0.01  0.08  0.33
X_36  0.24 -0.01  1.00 -0.53  0.28
X_42 -0.14  0.08 -0.53  1.00  0.27
X_28  0.19  0.33  0.28  0.27  1.00
💾 Cached to /content/drive/MyDrive/26-1 머신러닝 팀플/data/cache/core_sensors.json
📦 Load cached core sensors (k=5): ['X_40', 'X_46', 'X_36', 'X_42', 'X_28']
✅ FE 적용 완료 | train: (21693, 60), test: (15004, 59)

🚀 [1/5] Training TabNet (5-Fold)...
🛠️  Run & cache: tabnet
🧪 TabNet self-supervised pretraining (raw features)...


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)



🌲 TabNet Fold 1/5 | use_lda=True


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.29305 | val_accuracy: 0.32404 | val_logloss: 2.31315 |  0:00:04s
epoch 10 | loss: 1.02266 | val_accuracy: 0.59253 | val_logloss: 1.29737 |  0:00:39s
epoch 20 | loss: 0.80086 | val_accuracy: 0.69578 | val_logloss: 0.83983 |  0:01:15s
epoch 30 | loss: 0.67484 | val_accuracy: 0.74049 | val_logloss: 0.70186 |  0:01:51s
epoch 40 | loss: 0.58558 | val_accuracy: 0.67043 | val_logloss: 1.01885 |  0:02:27s
epoch 50 | loss: 0.50336 | val_accuracy: 0.78613 | val_logloss: 0.57003 |  0:03:05s
epoch 60 | loss: 0.45238 | val_accuracy: 0.80733 | val_logloss: 0.50994 |  0:03:40s
epoch 70 | loss: 0.45082 | val_accuracy: 0.75847 | val_logloss: 0.67517 |  0:04:16s
epoch 80 | loss: 0.3802  | val_accuracy: 0.77875 | val_logloss: 0.61521 |  0:04:52s
epoch 90 | loss: 0.36264 | val_accuracy: 0.85895 | val_logloss: 0.35649 |  0:05:27s
epoch 100| loss: 0.36442 | val_accuracy: 0.77875 | val_logloss: 0.65079 |  0:06:03s
epoch 110| loss: 0.37937 | val_accuracy: 0.82254 | val_logloss: 0.47311 |  0

...

  Fold 3: F1 = 0.8032


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 4: F1 = 0.7979
📊 LightGBM 평균 OOF F1: 0.7989

🚀 [5/5] Training AutoGluon...
🛠️  Run & cache: autogluon


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.09 GB / 12.67 GB (79.6%)
Disk Space Avail:   71.87 GB / 107.72 GB (66.7%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` 


⚖️ Auto-weight ensemble (4 models + AG)
🎯 Weights: TabNet=0.295 ReLU=0.243 GELU=0.461 LGB=0.001 | OOF F1=0.8782

✅ Final saved: /content/drive/MyDrive/머신러닝/data/submission_5models.csv (Hungarian used)

📊 클래스별 예측 개수:
0     714
1     714
2     714
3     715
4     715
5     714
6     714
7     714
8     715
9     714
10    715
11    715
12    715
13    715
14    715
15    714
16    714
17    715
18    714
19    714
20    715
Name: count, dtype: int64
